# 02 · Preprocessing and temporal split

Turns the EDA decisions into data the models can use. The logic lives in `src/musicrec/preprocess.py`;
this notebook runs it, picks the item filter threshold, and runs sanity checks.

| Split | Train | Evaluate on | Used for |
|---|---|---|---|
| `split_val`  | < 2009-01-01 | Jan–Feb 2009 | model selection, hyperparameters, preprocessing choices |
| `split_test` | < 2009-03-01 | Mar–Apr 2009 | final numbers, run once per model at the very end |

**Task:** recommend tracks the user has *never* played before. Repeats still matter as training
signal (50 plays = stronger preference than 1), but they are never evaluation targets.

In [1]:
import pandas as pd
from musicrec import config
from musicrec.data import load_listens
from musicrec.preprocess import clean, item_threshold_table, make_split

pd.set_option("display.float_format", "{:,.3f}".format)

events, tracks = clean(load_listens(columns=["user_id", "ts", "artist_name", "track_name"]))
tracks.to_parquet(config.TRACKS_PARQUET, index=False)
events.head()

clean: 19,150,868 rows -> 18,956,614 after time window -> 18,956,612 after dedup | 991 users | 1,489,200 tracks


,user_id,ts,track_key
0,user_000001,2006-08-13 13:59:20+00:00,1036
1,user_000001,2006-08-13 14:03:29+00:00,1035
2,user_000001,2006-08-13 14:10:43+00:00,1034
3,user_000001,2006-08-13 14:17:40+00:00,3123
4,user_000001,2006-08-13 14:19:06+00:00,3039


## Choosing the minimum-listeners threshold

Decided on the **validation** split only. Choosing it by looking at test results would be a
quiet form of leakage: the test set must not influence any decision.

In [2]:
table = item_threshold_table(events, config.VAL_TRAIN_END, config.VAL_EVAL_END)
table

265,647 (user, new track) pairs in the eval window; 28.7% are tracks nobody played during training (pure cold start)


,min_listeners,n_items,train listens kept,future discoveries recommendable
0,1,1344546,1.000,0.713
1,2,456854,0.825,0.620
2,3,278116,0.749,0.560
3,5,154662,0.657,0.479
4,10,70390,0.531,0.366
5,20,30122,0.399,0.254
6,50,7630,0.215,0.121


In [3]:
MIN_ITEM_USERS = 10   # revisit after looking at the table above
MIN_USER_ITEMS = 10

splits = {}
for name, (t0, t1), out in [
    ("val",  (config.VAL_TRAIN_END,  config.VAL_EVAL_END),  config.SPLIT_VAL_DIR),
    ("test", (config.TEST_TRAIN_END, config.TEST_EVAL_END), config.SPLIT_TEST_DIR),
]:
    s = make_split(events, t0, t1, MIN_ITEM_USERS, MIN_USER_ITEMS)
    s.save(out)
    splits[name] = s

pd.DataFrame({k: s.stats for k, s in splits.items()})

,val,test
n_users,922.000,954.000
n_items,"70,390.000","75,145.000"
train_listens,"8,824,538.000","9,575,373.000"
train_pairs,"1,834,370.000","1,982,972.000"
density,0.028,0.028
eval_new_pairs_all,"265,647.000","280,231.000"
eval_new_pairs_reachable,"88,001.000","92,635.000"
eval_users_with_targets,645.000,720.000


## Sanity checks (cheap insurance against silent bugs)

In [4]:
for name, s in splits.items():
    # 1. no target was already in the user's training history
    overlap = s.targets.merge(s.train_counts, on=["user_idx", "item_idx"])
    assert overlap.empty, f"{name}: {len(overlap)} targets leak from train"
    # 2. training data stops before the cutoff
    assert s.train_events["ts"].max() < pd.Timestamp(s.train_end, tz="UTC")
    # 3. indices are contiguous and in range
    assert s.train_counts["user_idx"].between(0, s.n_users - 1).all()
    assert s.train_counts["item_idx"].between(0, s.n_items - 1).all()
    assert s.targets["item_idx"].between(0, s.n_items - 1).all()
    print(f"{name}: all checks passed")

val: all checks passed
test: all checks passed


In [5]:
# Targets per evaluated user: how many discoveries does a typical user make in 2 months?
s = splits["val"]
s.targets.groupby("user_idx").size().describe(percentiles=[0.1, 0.5, 0.9])

count     645.000
mean      136.436
std       213.053
min         1.000
10%         9.000
50%        65.000
90%       344.000
max     2,286.000
dtype: float64